# 무드포트 지식봇 — 다층 평가 (BLEU · ROUGE · 임베딩 · LLM-as-Judge)
**평가 대상:** 4번에서 지식 주입한 `moodtree_lora` 어댑터 (베이스: Qwen2.5-7B-Instruct)
**환경:** Google Colab (T4 GPU)

---

### 이 노트북에서 배우는 것
앞선 노트북들에서는 모델을 **학습**시켰습니다. 이번엔 학습이 끝난 모델을 **어떻게 평가하는가**를 다룹니다.

좋은 답변인지 판단하는 방법은 한 가지가 아닙니다. **세 층위**로 나눠서 봅니다.

| 층위 | 방법 | 무엇을 보나 | 한계 |
|------|------|------------|------|
| ① 단어 수준 | **BLEU / ROUGE** | 정답과 **겹치는 단어/n-gram** 비율 | 같은 뜻 다른 표현이면 0점 |
| ② 의미 수준 | **임베딩 코사인 유사도** | 문장의 **의미**가 가까운가 | 사실 오류를 못 잡음 |
| ③ 판단 수준 | **LLM as Judge** | 사람처럼 **맥락·사실**을 채점 | 비용·judge 모델 편향 |

> 핵심 메시지: **단어가 안 겹쳐도 의미는 맞을 수 있고, 의미가 가까워도 사실은 틀릴 수 있습니다.**
> 그래서 한 지표만 믿으면 안 되고, 층을 쌓아 올려야 평가가 완성됩니다.

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [1]:
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets
!pip install -q evaluate rouge-score nltk sentence-transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


지저분한 경고 메세지를 띄우지 않기 위한 코드

In [2]:
import warnings
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

## 2. 학습된 모델 불러오기 — 베이스 모델 + LoRA 어댑터

학습 때 우리가 저장한 건 **LoRA 어댑터(약 150MB)뿐**입니다. 베이스 모델(Qwen2.5-7B)은 따로 받아서, 그 위에 어댑터를 얹습니다.

> ⚠️ 이 어댑터의 베이스는 **Qwen2.5-7B-Instruct** 입니다 (앞 노트북의 1.5B가 아님).
> 7B라 다운로드에 몇 분 걸리지만, 4bit 양자화로 T4 메모리 안에 충분히 들어갑니다.

### 어댑터 폴더 업로드
로컬의 `moodtree_lora` 폴더를 통째로 zip으로 압축해서 업로드하세요.
- 내 PC에서 `moodtree_lora` 폴더 → 우클릭 → "압축(zip)" → `moodtree_lora.zip` 생성
- 아래 셀 실행 후 그 zip을 선택

In [3]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # moodtree_lora.zip 선택

zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(".")

# 압축 안에 폴더가 한 겹 더 들어간 경우까지 대비해 adapter_config.json 위치를 찾는다
ADAPTER_DIR = None
for root, _, fnames in os.walk("."):
    if "adapter_config.json" in fnames:
        ADAPTER_DIR = root
        break

print("✅ 어댑터 경로:", ADAPTER_DIR)
assert ADAPTER_DIR is not None, "adapter_config.json을 찾지 못했습니다. zip 내용을 확인하세요."

Saving moodtree_lora.zip to moodtree_lora.zip
✅ 어댑터 경로: ./moodtree_lora


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# 베이스 모델 위에 학습된 LoRA 어댑터를 얹는다
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

print("✅ 학습된 모델 로드 완료")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ 학습된 모델 로드 완료


## 3. 평가용 질문·정답(gold) 준비

평가하려면 **모델의 답**과 비교할 **정답(gold)** 이 있어야 합니다.

> ⚠️ 중요: 여기 질문들은 **학습에 그대로 쓰이지 않은 표현(held-out)** 으로 바꿔서 냈습니다.
> 학습에 있던 문장을 똑같이 물으면 "외운 걸" 보는 것이고, 표현을 바꿔 물어야 **진짜 지식이 들어갔는지**를 봅니다.

각 항목은 `question`(질문)과 `reference`(모범 정답) 한 쌍입니다.

In [5]:
eval_data = [
    {"question": "무드포트에서 4만원어치 사면 배송비가 붙나요?",
     "reference": "4만원 이상이면 무료배송이라 배송비가 없어요."},
    {"question": "무드포트는 어느 도시에서 시작한 브랜드인가요?",
     "reference": "제주시에 본사를 둔 브랜드예요."},
    {"question": "무드포트 마스코트 캐릭터를 설명해줘.",
     "reference": "향고래를 모티프로 한 '포포'라는 캐릭터예요."},
    {"question": "디퓨저 하나 사면 향이 대략 얼마나 유지되나요?",
     "reference": "평균 약 8주 정도 지속돼요."},
    {"question": "무드포트 회원 등급은 총 몇 단계예요?",
     "reference": "데일리, 무드, 시그니처 3단계예요."},
    {"question": "가장 높은 멤버십 등급의 혜택을 알려줘.",
     "reference": "시그니처 등급은 전 상품 무료배송과 10% 적립 혜택이 있어요."},
    {"question": "무드포트 고객센터는 주말에도 문의할 수 있나요?",
     "reference": "주말과 공휴일은 휴무라 평일에만 문의할 수 있어요."},
    {"question": "매달 디퓨저를 받아보는 구독 서비스 이름이 뭐죠?",
     "reference": "'무드레터'예요."},
    {"question": "무드포트 신제품 출시 주기는 어떻게 되나요?",
     "reference": "분기마다, 1년에 네 번 출시돼요."},
    {"question": "무드포트의 시그니처 향 계열은 무엇인가요?",
     "reference": "산뜻한 시트러스에 부드러운 머스크를 더한 시트러스 머스크 계열이에요."},
    {"question": "상품 받고 며칠 안에 환불 신청해야 하나요?",
     "reference": "수령 후 14일 이내에 신청하셔야 해요."},
    {"question": "무드포트 포장은 환경을 고려하나요?",
     "reference": "네, 100% 재생지를 사용한 친환경 포장이에요."},
]

print(f"✅ 평가 질문 {len(eval_data)}개 준비 완료")

✅ 평가 질문 12개 준비 완료


## 4. 모델 답변 생성

평가 질문을 모델에 넣어 답을 받아 `records`에 모읍니다. 채점은 **재현 가능해야** 하므로 `do_sample=False`(그리디 디코딩)로 고정합니다.

In [6]:
import torch

def generate(prompt, max_new_tokens=160):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=False,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

records = []
for ex in eval_data:
    pred = generate(ex["question"])
    records.append({
        "question":  ex["question"],
        "reference": ex["reference"],
        "prediction": pred,
    })
    print(f"Q: {ex['question']}")
    print(f"A(모델): {pred}")
    print(f"A(정답): {ex['reference']}")
    print("-" * 60)

Q: 무드포트에서 4만원어치 사면 배송비가 붙나요?
A(모델): 네, 4만원 미만이라 배송비가 부과돼요.
A(정답): 4만원 이상이면 무료배송이라 배송비가 없어요.
------------------------------------------------------------
Q: 무드포트는 어느 도시에서 시작한 브랜드인가요?
A(모델): 제주시에서 시작한 브랜드예요.
A(정답): 제주시에 본사를 둔 브랜드예요.
------------------------------------------------------------
Q: 무드포트 마스코트 캐릭터를 설명해줘.
A(모델): 무드포트의 마스코트는 '포포'입니다. 향고래 모티프를 디퓨저 브랜드에 어울리게 재해석한 귀여운 캐릭터로, 고객들에게 향·디퓨저에 대한 지식을 전달하고 디퓨저를 통해 편안한 휴식을 즐길 수 있다는 메시지를 전달합니다.
A(정답): 향고래를 모티프로 한 '포포'라는 캐릭터예요.
------------------------------------------------------------
Q: 디퓨저 하나 사면 향이 대략 얼마나 유지되나요?
A(모델): 평균 6주 정도 향이 지속돼요.
A(정답): 평균 약 8주 정도 지속돼요.
------------------------------------------------------------
Q: 무드포트 회원 등급은 총 몇 단계예요?
A(모델): 4단계예요.
A(정답): 데일리, 무드, 시그니처 3단계예요.
------------------------------------------------------------
Q: 가장 높은 멤버십 등급의 혜택을 알려줘.
A(모델): 최상위 등급인 '시그니처'는 전 상품 무료배송과 10% 적립 혜택이 있어요.
A(정답): 시그니처 등급은 전 상품 무료배송과 10% 적립 혜택이 있어요.
------------------------------------------------------------
Q: 무드포트 고객센터는 

## 5. 단어 수준 평가 — BLEU & ROUGE

가장 고전적인 자동 평가입니다. **정답과 모델 답이 단어/구를 얼마나 공유하는가**를 봅니다.

- **ROUGE-L**: 정답과 답변의 **가장 긴 공통 부분 수열(LCS)** 기반. 재현율 성격(정답을 얼마나 담았나).
- **BLEU**: 답변의 **n-gram이 정답에 얼마나 들어맞나**(정밀도 성격). 짧은 문장엔 가혹합니다.

> 한국어는 띄어쓰기 단위 토큰화라 어미만 달라도 점수가 뚝 떨어집니다. 이 "가혹함" 자체가 6번과 비교할 포인트예요.

In [12]:
import evaluate

rouge_metric = evaluate.load("rouge")
bleu_metric  = evaluate.load("bleu")

for r in records:
    rouge = rouge_metric.compute(
        predictions=[r["prediction"]], references=[r["reference"]]
    )
    try:
        bleu = bleu_metric.compute(
            predictions=[r["prediction"]], references=[[r["reference"]]]
        )["bleu"]
    except ZeroDivisionError:
        bleu = 0.0
    r["rougeL"] = round(float(rouge["rougeL"]), 3)
    r["bleu"]   = round(float(bleu), 3)

print(f"{'정답(reference)':30s} {'생성결과(prediction)':30s} {'ROUGE-L':>8s} {'BLEU':>8s}")
print("-" * 80)
for r in records:
    print(f"{r['reference'][:28]:30s} {r['prediction'][:28]:30s} {r['rougeL']:>8.3f} {r['bleu']:>8.3f}")

avg_rouge = sum(r["rougeL"] for r in records) / len(records)
avg_bleu  = sum(r["bleu"]  for r in records) / len(records)
print("-" * 80)
print(f"{'':30s} {'평균':30s} {avg_rouge:>8.3f} {avg_bleu:>8.3f}")

정답(reference)                  생성결과(prediction)                ROUGE-L     BLEU
--------------------------------------------------------------------------------
4만원 이상이면 무료배송이라 배송비가 없어요.      네, 4만원 미만이라 배송비가 부과돼요.            1.000    0.000
제주시에 본사를 둔 브랜드예요.              제주시에서 시작한 브랜드예요.                  0.000    0.000
향고래를 모티프로 한 '포포'라는 캐릭터예요.      무드포트의 마스코트는 '포포'입니다. 향고래 모티프      0.000    0.000
평균 약 8주 정도 지속돼요.               평균 6주 정도 향이 지속돼요.                 0.000    0.000
데일리, 무드, 시그니처 3단계예요.           4단계예요.                            0.000    0.000
시그니처 등급은 전 상품 무료배송과 10% 적립 혜   최상위 등급인 '시그니처'는 전 상품 무료배송과 1      1.000    0.710
주말과 공휴일은 휴무라 평일에만 문의할 수 있어요.   아니요, 고객센터는 주말과 공휴일은 휴무라 평일에       0.000    0.000
'무드레터'예요.                      '무드레터'예요.                         0.000    0.000
분기마다, 1년에 네 번 출시돼요.            분기마다, 1년에 네 번 신제품을 출시합니다.         1.000    0.541
산뜻한 시트러스에 부드러운 머스크를 더한 시트러스    시트러스 머스크 계열이 시그니처예요.              0.000    0.000
수령 후 14일 이내에 신청하셔야 해요.         상품 수령 후 

## 6. 의미 수준 평가 — 임베딩 코사인 유사도

BLEU/ROUGE의 약점은 분명합니다. "제주시에 있어요" vs "본사는 제주에 위치합니다" 는 **뜻이 같은데 단어가 안 겹쳐서** 점수가 낮게 나옵니다.

그래서 문장을 **임베딩 벡터**로 바꿔, 벡터 사이의 **코사인 유사도(0~1)** 로 *의미가* 가까운지를 봅니다.
한국어 문장 임베딩 모델 `jhgan/ko-sroberta-multitask` 를 사용합니다.

In [13]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("jhgan/ko-sroberta-multitask")

for r in records:
    emb = embedder.encode(
        [r["prediction"], r["reference"]],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    r["cosine"] = round(float(util.cos_sim(emb[0], emb[1])), 3)

print(f"{'정답(reference)':30s} {'생성결과(prediction)':30s} {'ROUGE-L':>8s} {'BLEU':>8s} {'임베딩':>8s}")
print("-" * 90)
for r in records:
    print(f"{r['reference'][:28]:30s} {r['prediction'][:28]:30s} {r['rougeL']:>8.3f} {r['bleu']:>8.3f} {r['cosine']:>8.3f}")

avg_cos = sum(r["cosine"] for r in records) / len(records)
print("-" * 90)
print(f"{'':30s} {'평균':30s} {avg_rouge:>8.3f} {avg_bleu:>8.3f} {avg_cos:>8.3f}")

정답(reference)                  생성결과(prediction)                ROUGE-L     BLEU      임베딩
------------------------------------------------------------------------------------------
4만원 이상이면 무료배송이라 배송비가 없어요.      네, 4만원 미만이라 배송비가 부과돼요.            1.000    0.000    0.687
제주시에 본사를 둔 브랜드예요.              제주시에서 시작한 브랜드예요.                  0.000    0.000    0.910
향고래를 모티프로 한 '포포'라는 캐릭터예요.      무드포트의 마스코트는 '포포'입니다. 향고래 모티프      0.000    0.000    0.784
평균 약 8주 정도 지속돼요.               평균 6주 정도 향이 지속돼요.                 0.000    0.000    0.540
데일리, 무드, 시그니처 3단계예요.           4단계예요.                            0.000    0.000    0.449
시그니처 등급은 전 상품 무료배송과 10% 적립 혜   최상위 등급인 '시그니처'는 전 상품 무료배송과 1      1.000    0.710    0.946
주말과 공휴일은 휴무라 평일에만 문의할 수 있어요.   아니요, 고객센터는 주말과 공휴일은 휴무라 평일에       0.000    0.000    0.811
'무드레터'예요.                      '무드레터'예요.                         0.000    0.000    1.000
분기마다, 1년에 네 번 출시돼요.            분기마다, 1년에 네 번 신제품을 출시합니다.         1.000    0.541    0.928
산뜻한 시트러스에 부드러운 머스크를

👉 **BLEU는 0점인데 임베딩은 0.8이 넘는 줄**을 찾아보세요. 단어는 안 겹쳐도 **의미는 정답과 거의 같다**는 뜻입니다.

> 단, 임베딩 유사도도 만능이 아닙니다. "**2021년** 설립"을 "**2019년** 설립"이라 답해도, 문장 구조가 비슷하면 코사인은 높게 나옵니다.
> → **숫자·사실 오류는 의미 유사도로 못 잡습니다.** 그래서 마지막 층(LLM judge)이 필요합니다.

## 7. 결과를 JSON으로 저장

지금까지의 질문·정답·모델답·점수를 한 파일(`moodtree_eval_results.json`)로 저장합니다.
이 파일이 다음 단계(LLM judge)의 **입력**이 됩니다.

In [10]:
import json

with open("moodtree_eval_results.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료 → moodtree_eval_results.json")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

✅ 저장 완료 → moodtree_eval_results.json
{
  "question": "무드포트에서 4만원어치 사면 배송비가 붙나요?",
  "reference": "4만원 이상이면 무료배송이라 배송비가 없어요.",
  "prediction": "네, 4만원 미만이라 배송비가 부과돼요.",
  "rougeL": 1.0,
  "bleu": 0.0,
  "cosine": 0.687
}


## 8. LLM as Judge — 챗봇에게 채점을 맡기기

자동 지표(BLEU/ROUGE/임베딩)는 **사실이 맞는지**는 판단하지 못합니다.
사람이 일일이 읽는 대신, **성능 좋은 LLM에게 채점관 역할**을 맡기는 방식이 LLM-as-Judge입니다.

여기서는 방금 저장한 JSON을 통째로 judge 모델에게 주고, 각 항목을 **PASS/FAIL + 1~5점 + 이유**로 채점하게 합니다.

> 토큰 입력: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 에서 토큰을 만들어 아래 `login()` 실행 시 붙여넣으세요.

In [11]:
from huggingface_hub import login, InferenceClient
import json

login()  # HF 토큰 입력

# judge로 쓸 모델 (더 강한 모델일수록 채점이 정확합니다. 필요시 교체)
JUDGE_MODEL = "Qwen/Qwen2.5-72B-Instruct"
client = InferenceClient(model=JUDGE_MODEL)

with open("moodtree_eval_results.json", encoding="utf-8") as f:
    saved = json.load(f)

judge_items = [
    {"id": i, "question": r["question"], "reference": r["reference"], "answer": r["prediction"]}
    for i, r in enumerate(saved)
]

system_prompt = (
    "당신은 고객상담 챗봇의 답변을 채점하는 엄격하고 공정한 평가자입니다. "
    "reference(모범 정답)에 담긴 사실과 answer(모델 답변)의 사실이 일치하는지를 기준으로 채점하세요. "
    "표현이 달라도 사실이 맞으면 PASS, 숫자나 사실이 틀리거나 빠지면 FAIL입니다."
)

user_prompt = (
    "아래 평가 항목들을 채점하세요. 반드시 다음 JSON 배열 형식으로만 출력하세요(설명 문장 금지):\n"
    '[{"id": 0, "verdict": "PASS 또는 FAIL", "score": 1~5 정수, "reason": "한 줄 이유"}, ...]\n\n'
    "데이터:\n" + json.dumps(judge_items, ensure_ascii=False, indent=2)
)

try:
    resp = client.chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=1500,
        temperature=0.0,
    )
    judge_raw = resp.choices[0].message.content
    print(judge_raw)
except Exception as e:
    judge_raw = None
    print("⚠️ judge 모델 호출 실패:", e)
    print("→ 아래 9번 셀의 '수동 채점' 방법을 사용하세요. (JSON은 이미 저장돼 있습니다)")

⚠️ judge 모델 호출 실패: You must provide an api_key to work with auto API or log in with `hf auth login`.
→ 아래 9번 셀의 '수동 채점' 방법을 사용하세요. (JSON은 이미 저장돼 있습니다)


In [ ]:
import json, re

if judge_raw:
    # judge가 출력한 텍스트에서 JSON 배열만 추출
    m = re.search(r"\[.*\]", judge_raw, re.DOTALL)
    verdicts = json.loads(m.group(0)) if m else []
    by_id = {v["id"]: v for v in verdicts}

    n_pass = sum(1 for v in verdicts if str(v.get("verdict", "")).upper().startswith("PASS"))
    print(f"{'질문':30s} {'판정':>6s} {'점수':>5s}  이유")
    print("-" * 80)
    for i, r in enumerate(saved):
        v = by_id.get(i, {})
        mark = "✅" if str(v.get("verdict", "")).upper().startswith("PASS") else "❌"
        print(f"{r['question'][:28]:30s} {mark:>6s} {str(v.get('score','-')):>5s}  {v.get('reason','')}")
    print("-" * 80)
    print(f"\n📊 LLM Judge 종합: PASS {n_pass}/{len(saved)} ({n_pass/len(saved)*100:.0f}%)")
else:
    print("judge 출력이 없습니다. 9번 수동 채점을 사용하세요.")

## 9. (대안) 챗봇에 직접 붙여넣어 채점받기

위 API 호출이 안 되면, 저장된 `moodtree_eval_results.json` 을 내려받아 ChatGPT·Claude 같은 챗봇 창에 붙여넣고 아래 프롬프트로 채점받을 수 있습니다. **이것도 똑같은 LLM-as-Judge입니다.**

In [ ]:
from google.colab import files

print("아래 프롬프트를 복사 → 챗봇에 붙여넣고, 이어서 다운로드한 JSON을 붙여넣으세요.\n")
print("=" * 60)
print(
    "너는 고객상담 챗봇 답변을 채점하는 평가자야.\n"
    "아래 JSON의 각 항목에서 reference(정답)와 answer(모델 답변)의 사실이 일치하면 PASS, "
    "숫자나 사실이 틀리면 FAIL로 판정하고, 1~5점과 한 줄 이유를 붙여 표로 정리해줘.\n"
    "표현이 달라도 사실이 맞으면 PASS야.\n\n"
    "[여기에 moodtree_eval_results.json 내용 붙여넣기]"
)
print("=" * 60)

files.download("moodtree_eval_results.json")

## 10. 정리 — 평가는 한 층으로 끝나지 않는다

| 층위 | 잡아내는 것 | 못 잡는 것 |
|------|------------|-----------|
| BLEU / ROUGE | 표현이 정답과 똑같은가 | 같은 뜻 다른 표현 (→ 과소평가) |
| 임베딩 유사도 | 의미가 정답과 가까운가 | 숫자·사실 오류 (→ 놓침) |
| LLM as Judge | 사실·맥락이 맞는가 | 비용, judge 모델 자체의 편향 |

> **실무 가이드**
> - 빠른 자동 회귀 테스트(CI)에는 BLEU/ROUGE·임베딩 유사도를 쓰고,
> - 출시 전 품질 판정이나 안전성 검수에는 LLM judge(또는 사람)를 씁니다.
> - 어떤 지표도 단독으로는 충분하지 않습니다. **여러 층을 겹쳐** 보는 것이 핵심입니다.